# Klasifikasi Jenis Sampah di Sekitar Kampus dengan CNN
**Nama  :** [Nama Anda]  
**NIM   :** [NIM Anda]

Notebook ini mengimplementasikan **Convolutional Neural Network (CNN)** untuk mengklasifikasikan jenis sampah di sekitar kampus menggunakan **Trash Type Image Dataset** dari Kaggle.

Dataset terdiri dari gambar sampah yang dikategorikan ke dalam beberapa kelas seperti: *cardboard*, *glass*, *metal*, *paper*, *plastic*, dan *trash*.

---

## 1. Import Library

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.metrics import classification_report, confusion_matrix

# Pengaturan tampilan
plt.style.use('ggplot')
sns.set_theme(style="whitegrid")
sns.set_palette("husl")

print(f"TensorFlow version : {tf.__version__}")
print(f"Keras version      : {keras.__version__}")
print(f"GPU tersedia       : {tf.config.list_physical_devices('GPU')}")
print("Library berhasil diimpor ✓")

ModuleNotFoundError: No module named 'tensorflow'

## 2. Konfigurasi Dataset

Dataset diunduh dari Kaggle: [Trash Type Image Dataset](https://www.kaggle.com/datasets/farzadnekouei/trash-type-image-dataset)

Pastikan folder dataset sudah tersedia. Struktur folder yang diharapkan:
```
trash-type-image-dataset/
    ├── cardboard/
    ├── glass/
    ├── metal/
    ├── paper/
    ├── plastic/
    └── trash/
```

In [ ]:
# ── Konfigurasi Kaggle API (jalankan sekali) ────────────────────────────────
# from google.colab import files
# files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# ── Download Dataset ─────────────────────────────────────────────────────────
# !kaggle datasets download -d farzadnekouei/trash-type-image-dataset
# !unzip -q trash-type-image-dataset.zip -d trash-dataset

# ── Path Dataset ─────────────────────────────────────────────────────────────
DATASET_DIR = 'trash-dataset'   # Sesuaikan dengan path Anda

# Parameter
IMG_SIZE    = (224, 224)         # Ukuran gambar input
BATCH_SIZE  = 32
EPOCHS      = 30
SEED        = 42

# Deteksi kelas dari folder
CLASS_NAMES = sorted([
    d for d in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, d))
])
NUM_CLASSES = len(CLASS_NAMES)

print(f"Kelas yang ditemukan  : {CLASS_NAMES}")
print(f"Jumlah kelas          : {NUM_CLASSES}")
print(f"Ukuran gambar input   : {IMG_SIZE}")
print(f"Batch size            : {BATCH_SIZE}")

## 3. Exploratory Data Analysis (EDA)

Sebelum membangun model, kita perlu memahami distribusi data dan visualisasi contoh gambar.

In [ ]:
# ── Hitung jumlah gambar per kelas ───────────────────────────────────────────
kelas_info = {}
total_gambar = 0

for kelas in CLASS_NAMES:
    path_kelas = os.path.join(DATASET_DIR, kelas)
    ekstensi   = ('.jpg', '.jpeg', '.png', '.webp')
    jumlah     = sum(
        1 for f in os.listdir(path_kelas)
        if f.lower().endswith(ekstensi)
    )
    kelas_info[kelas] = jumlah
    total_gambar += jumlah

df_eda = pd.DataFrame(list(kelas_info.items()), columns=['Kelas', 'Jumlah Gambar'])
df_eda['Persentase (%)'] = (df_eda['Jumlah Gambar'] / total_gambar * 100).round(2)
df_eda = df_eda.sort_values('Jumlah Gambar', ascending=False).reset_index(drop=True)

print(f"Total gambar   : {total_gambar}")
print(f"Rata-rata/kelas: {total_gambar // NUM_CLASSES}")
print()
print(df_eda.to_string(index=False))

In [ ]:
# ── Visualisasi distribusi kelas ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Distribusi Dataset Sampah', fontsize=14, fontweight='bold')

# Bar chart
axes[0].bar(df_eda['Kelas'], df_eda['Jumlah Gambar'], color=sns.color_palette('husl', NUM_CLASSES))
axes[0].set_title('Jumlah Gambar per Kelas')
axes[0].set_xlabel('Kelas Sampah')
axes[0].set_ylabel('Jumlah Gambar')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(df_eda['Jumlah Gambar']):
    axes[0].text(i, v + 5, str(v), ha='center', fontsize=9, fontweight='bold')

# Pie chart
axes[1].pie(df_eda['Jumlah Gambar'], labels=df_eda['Kelas'],
            autopct='%1.1f%%', colors=sns.color_palette('husl', NUM_CLASSES),
            startangle=90)
axes[1].set_title('Proporsi Kelas Dataset')

plt.tight_layout()
plt.show()

In [ ]:
# ── Tampilkan contoh gambar dari setiap kelas ────────────────────────────────
from tensorflow.keras.preprocessing.image import load_img

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
fig.suptitle('Contoh Gambar per Kelas Sampah', fontsize=14, fontweight='bold')

for ax, kelas in zip(axes.flatten(), CLASS_NAMES[:6]):
    path_kelas  = os.path.join(DATASET_DIR, kelas)
    contoh_file = [
        f for f in os.listdir(path_kelas)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ][0]
    img = load_img(os.path.join(path_kelas, contoh_file), target_size=IMG_SIZE)
    ax.imshow(img)
    ax.set_title(kelas.upper(), fontsize=11, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 4. Preprocessing & Augmentasi Data

Langkah preprocessing meliputi:
1. **Normalisasi** — piksel dinormalisasi ke rentang [0, 1]
2. **Train-Validation-Test Split** — 70% training, 15% validasi, 15% testing
3. **Data Augmentation** — rotasi, flip, zoom untuk memperkaya data training

In [ ]:
# ── Data Augmentation & Normalisasi ──────────────────────────────────────────
train_datagen = ImageDataGenerator(
    rescale           = 1.0 / 255,       # Normalisasi
    validation_split  = 0.2,             # 20% untuk validasi
    rotation_range    = 20,
    width_shift_range = 0.15,
    height_shift_range= 0.15,
    shear_range       = 0.10,
    zoom_range        = 0.15,
    horizontal_flip   = True,
    fill_mode         = 'nearest'
)

val_datagen = ImageDataGenerator(
    rescale          = 1.0 / 255,
    validation_split = 0.2
)

# ── Data Generator ────────────────────────────────────────────────────────────
train_generator = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size  = IMG_SIZE,
    batch_size   = BATCH_SIZE,
    class_mode   = 'categorical',
    subset       = 'training',
    seed         = SEED
)

val_generator = val_datagen.flow_from_directory(
    DATASET_DIR,
    target_size  = IMG_SIZE,
    batch_size   = BATCH_SIZE,
    class_mode   = 'categorical',
    subset       = 'validation',
    seed         = SEED,
    shuffle      = False
)

print(f"\nData training  : {train_generator.samples} gambar")
print(f"Data validasi  : {val_generator.samples} gambar")
print(f"Kelas mapping  : {train_generator.class_indices}")

In [ ]:
# ── Visualisasi Augmentasi ────────────────────────────────────────────────────
aug_sample_gen = ImageDataGenerator(
    rescale           = 1.0 / 255,
    rotation_range    = 20,
    width_shift_range = 0.15,
    height_shift_range= 0.15,
    zoom_range        = 0.15,
    horizontal_flip   = True,
    fill_mode         = 'nearest'
)

# Ambil satu contoh gambar
sample_path  = os.path.join(DATASET_DIR, CLASS_NAMES[0])
sample_file  = [f for f in os.listdir(sample_path) if f.lower().endswith(('.jpg','.jpeg','.png'))][0]
sample_img   = load_img(os.path.join(sample_path, sample_file), target_size=IMG_SIZE)
sample_arr   = tf.keras.preprocessing.image.img_to_array(sample_img).reshape(1, *IMG_SIZE, 3)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle(f'Contoh Data Augmentation (Kelas: {CLASS_NAMES[0]})', fontsize=12, fontweight='bold')
aug_iter = aug_sample_gen.flow(sample_arr, batch_size=1)
for ax in axes.flatten():
    aug_img = next(aug_iter)[0]
    ax.imshow(aug_img)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Membangun Arsitektur CNN

Model CNN dibangun dari awal dengan arsitektur:
- **3 blok Convolutional** (Conv2D → BatchNorm → ReLU → MaxPool)
- **Global Average Pooling** untuk mengurangi dimensi
- **Dense layers** dengan Dropout untuk regularisasi
- **Softmax** di layer output

In [ ]:
def build_cnn(input_shape=(224, 224, 3), num_classes=6):
    """
    Membangun arsitektur CNN kustom untuk klasifikasi gambar sampah.
    """
    inputs = keras.Input(shape=input_shape, name='input_gambar')

    # ── Blok 1: 32 filter ────────────────────────────────────────────────────
    x = layers.Conv2D(32, (3, 3), padding='same', name='conv1_1')(inputs)
    x = layers.BatchNormalization(name='bn1_1')(x)
    x = layers.Activation('relu', name='relu1_1')(x)
    x = layers.Conv2D(32, (3, 3), padding='same', name='conv1_2')(x)
    x = layers.BatchNormalization(name='bn1_2')(x)
    x = layers.Activation('relu', name='relu1_2')(x)
    x = layers.MaxPooling2D((2, 2), name='pool1')(x)
    x = layers.Dropout(0.25, name='drop1')(x)

    # ── Blok 2: 64 filter ────────────────────────────────────────────────────
    x = layers.Conv2D(64, (3, 3), padding='same', name='conv2_1')(x)
    x = layers.BatchNormalization(name='bn2_1')(x)
    x = layers.Activation('relu', name='relu2_1')(x)
    x = layers.Conv2D(64, (3, 3), padding='same', name='conv2_2')(x)
    x = layers.BatchNormalization(name='bn2_2')(x)
    x = layers.Activation('relu', name='relu2_2')(x)
    x = layers.MaxPooling2D((2, 2), name='pool2')(x)
    x = layers.Dropout(0.25, name='drop2')(x)

    # ── Blok 3: 128 filter ───────────────────────────────────────────────────
    x = layers.Conv2D(128, (3, 3), padding='same', name='conv3_1')(x)
    x = layers.BatchNormalization(name='bn3_1')(x)
    x = layers.Activation('relu', name='relu3_1')(x)
    x = layers.Conv2D(128, (3, 3), padding='same', name='conv3_2')(x)
    x = layers.BatchNormalization(name='bn3_2')(x)
    x = layers.Activation('relu', name='relu3_2')(x)
    x = layers.MaxPooling2D((2, 2), name='pool3')(x)
    x = layers.Dropout(0.25, name='drop3')(x)

    # ── Classifier Head ───────────────────────────────────────────────────────
    x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = layers.Dense(256, activation='relu', name='fc1')(x)
    x = layers.BatchNormalization(name='bn_fc')(x)
    x = layers.Dropout(0.5, name='drop_fc')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='output')(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name='CNN_Sampah')
    return model

# Bangun model
model = build_cnn(input_shape=(*IMG_SIZE, 3), num_classes=NUM_CLASSES)
model.summary()

In [ ]:
# ── Visualisasi Arsitektur ────────────────────────────────────────────────────
keras.utils.plot_model(
    model,
    to_file     = 'arsitektur_cnn.png',
    show_shapes = True,
    show_layer_names = True,
    dpi         = 96
)
from IPython.display import Image as IPImage
IPImage('arsitektur_cnn.png')

## 6. Kompilasi & Pelatihan Model

Model dikompilasi menggunakan:
- **Optimizer**: Adam dengan learning rate adaptif
- **Loss function**: Categorical Cross-Entropy
- **Metrik**: Accuracy

Callbacks yang digunakan:
- **EarlyStopping** — menghentikan training jika validasi tidak membaik
- **ReduceLROnPlateau** — menurunkan learning rate saat plateau
- **ModelCheckpoint** — menyimpan model terbaik

In [ ]:
# ── Kompilasi ─────────────────────────────────────────────────────────────────
model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=1e-3),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

# ── Callbacks ─────────────────────────────────────────────────────────────────
callbacks = [
    EarlyStopping(
        monitor   = 'val_loss',
        patience  = 8,
        verbose   = 1,
        restore_best_weights = True
    ),
    ReduceLROnPlateau(
        monitor  = 'val_loss',
        factor   = 0.5,
        patience = 4,
        min_lr   = 1e-6,
        verbose  = 1
    ),
    ModelCheckpoint(
        filepath          = 'model_cnn_sampah_terbaik.h5',
        monitor           = 'val_accuracy',
        save_best_only    = True,
        verbose           = 1
    )
]

print("Model siap dilatih ✓")
print(f"Total parameter: {model.count_params():,}")

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────────
history = model.fit(
    train_generator,
    epochs            = EPOCHS,
    validation_data   = val_generator,
    callbacks         = callbacks,
    verbose           = 1
)

print(f"\nTraining selesai pada epoch ke-{len(history.history['loss'])}")
print(f"Akurasi training terbaik  : {max(history.history['accuracy'])*100:.2f}%")
print(f"Akurasi validasi terbaik  : {max(history.history['val_accuracy'])*100:.2f}%")

## 7. Kurva Training & Validasi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Kurva Training CNN — Klasifikasi Sampah', fontsize=14, fontweight='bold')

epochs_range = range(1, len(history.history['loss']) + 1)

# Loss
axes[0].plot(epochs_range, history.history['loss'],     label='Train Loss',      linewidth=2)
axes[0].plot(epochs_range, history.history['val_loss'], label='Validasi Loss',   linewidth=2, linestyle='--')
axes[0].set_title('Loss per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(epochs_range, history.history['accuracy'],     label='Train Accuracy',    linewidth=2)
axes[1].plot(epochs_range, history.history['val_accuracy'], label='Validasi Accuracy', linewidth=2, linestyle='--')
axes[1].set_title('Akurasi per Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Akurasi')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 8. Evaluasi Model pada Data Validasi

In [ ]:
# ── Prediksi ─────────────────────────────────────────────────────────────────
val_generator.reset()
y_pred_prob = model.predict(val_generator, verbose=1)
y_pred      = np.argmax(y_pred_prob, axis=1)
y_true      = val_generator.classes

# ── Akurasi keseluruhan ───────────────────────────────────────────────────────
from sklearn.metrics import accuracy_score
akurasi = accuracy_score(y_true, y_pred)
print(f"\nAkurasi pada Data Validasi: {akurasi * 100:.2f}%")

## 9. Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax
)
ax.set_title(f'Confusion Matrix — CNN Klasifikasi Sampah\nAkurasi: {akurasi*100:.2f}%',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Prediksi',  fontsize=11)
ax.set_ylabel('Aktual',    fontsize=11)
plt.xticks(rotation=30)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 10. Laporan Klasifikasi

In [ ]:
print("=" * 60)
print("   Laporan Klasifikasi — CNN Deteksi Jenis Sampah")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

## 11. Akurasi per Kelas

In [ ]:
# ── Akurasi per kelas ────────────────────────────────────────────────────────
cm_norm   = cm.astype('float') / cm.sum(axis=1, keepdims=True)
per_class = cm_norm.diagonal()

df_class = pd.DataFrame({
    'Kelas'          : CLASS_NAMES,
    'Akurasi (%)'    : (per_class * 100).round(2),
    'Jumlah Sampel'  : [np.sum(y_true == i) for i in range(NUM_CLASSES)]
}).sort_values('Akurasi (%)', ascending=False).reset_index(drop=True)

print("Akurasi per Kelas:")
print(df_class.to_string(index=False))

# Visualisasi
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(df_class['Kelas'], df_class['Akurasi (%)'],
              color=sns.color_palette('husl', NUM_CLASSES))
ax.set_title('Akurasi per Kelas Sampah', fontsize=13, fontweight='bold')
ax.set_xlabel('Kelas Sampah')
ax.set_ylabel('Akurasi (%)')
ax.set_ylim(0, 110)
ax.axhline(y=akurasi*100, color='red', linestyle='--', label=f'Rata-rata: {akurasi*100:.1f}%')
ax.legend()
for bar, val in zip(bars, df_class['Akurasi (%)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

## 12. Prediksi Contoh Gambar

In [ ]:
# ── Tampilkan 12 prediksi acak ───────────────────────────────────────────────
import random
from tensorflow.keras.preprocessing.image import load_img, img_to_array

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle('Prediksi CNN pada Contoh Gambar', fontsize=14, fontweight='bold')

for ax in axes.flatten():
    # Pilih kelas dan gambar acak
    kelas_dipilih = random.choice(CLASS_NAMES)
    path_kelas    = os.path.join(DATASET_DIR, kelas_dipilih)
    file_dipilih  = random.choice([
        f for f in os.listdir(path_kelas)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])

    img_path = os.path.join(path_kelas, file_dipilih)
    img      = load_img(img_path, target_size=IMG_SIZE)
    img_arr  = img_to_array(img) / 255.0

    pred_prob  = model.predict(img_arr[np.newaxis, ...], verbose=0)[0]
    pred_kelas = CLASS_NAMES[np.argmax(pred_prob)]
    confidence = pred_prob.max() * 100

    warna = 'green' if pred_kelas == kelas_dipilih else 'red'
    ax.imshow(img)
    ax.set_title(
        f"Aktual: {kelas_dipilih}\nPrediksi: {pred_kelas} ({confidence:.1f}%)",
        fontsize=8, color=warna, fontweight='bold'
    )
    ax.axis('off')

plt.tight_layout()
plt.show()

## 13. (Bonus) Transfer Learning dengan MobileNetV2

Sebagai perbandingan, kita coba gunakan arsitektur **MobileNetV2** yang sudah di-pretrain pada ImageNet untuk melihat apakah performa meningkat.

In [ ]:
# ── MobileNetV2 sebagai Base Model ───────────────────────────────────────────
base_model = keras.applications.MobileNetV2(
    input_shape = (*IMG_SIZE, 3),
    include_top = False,
    weights     = 'imagenet'
)
base_model.trainable = False   # Freeze base layers

# ── Tambahkan Classifier Head ─────────────────────────────────────────────────
inputs_tl = keras.Input(shape=(*IMG_SIZE, 3))
x_tl = keras.applications.mobilenet_v2.preprocess_input(inputs_tl)
x_tl = base_model(x_tl, training=False)
x_tl = layers.GlobalAveragePooling2D()(x_tl)
x_tl = layers.Dense(256, activation='relu')(x_tl)
x_tl = layers.Dropout(0.5)(x_tl)
outputs_tl = layers.Dense(NUM_CLASSES, activation='softmax')(x_tl)

model_tl = keras.Model(inputs_tl, outputs_tl, name='MobileNetV2_Sampah')
model_tl.compile(
    optimizer = keras.optimizers.Adam(1e-3),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)
model_tl.summary()

In [ ]:
# ── Re-buat generator untuk transfer learning ────────────────────────────────
tl_datagen = ImageDataGenerator(
    validation_split  = 0.2,
    rotation_range    = 20,
    width_shift_range = 0.15,
    height_shift_range= 0.15,
    zoom_range        = 0.15,
    horizontal_flip   = True,
    fill_mode         = 'nearest'
)

tl_train_gen = tl_datagen.flow_from_directory(
    DATASET_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', seed=SEED
)
tl_val_gen = tl_datagen.flow_from_directory(
    DATASET_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', seed=SEED, shuffle=False
)

# ── Training fase 1: Hanya head ───────────────────────────────────────────────
history_tl = model_tl.fit(
    tl_train_gen,
    epochs          = 15,
    validation_data = tl_val_gen,
    callbacks       = [
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
    ],
    verbose = 1
)

# ── Evaluasi Transfer Learning ────────────────────────────────────────────────
tl_val_gen.reset()
y_pred_tl    = np.argmax(model_tl.predict(tl_val_gen, verbose=0), axis=1)
akurasi_tl   = accuracy_score(tl_val_gen.classes, y_pred_tl)
print(f"\nAkurasi MobileNetV2 (Transfer Learning): {akurasi_tl * 100:.2f}%")

## 14. Perbandingan CNN Kustom vs Transfer Learning

In [ ]:
df_perbandingan = pd.DataFrame({
    'Model'               : ['CNN Kustom', 'MobileNetV2 (TL)'],
    'Akurasi Validasi (%)': [round(akurasi * 100, 2), round(akurasi_tl * 100, 2)],
    'Parameter'           : [model.count_params(), model_tl.count_params()],
    'Keterangan'          : ['Dibangun dari awal', 'Pretrained ImageNet']
})

print("Tabel Perbandingan Model:")
try:
    display(
        df_perbandingan.style
        .format({'Akurasi Validasi (%)': '{:.2f}', 'Parameter': '{:,}'})
        .background_gradient(cmap='YlGnBu', subset=['Akurasi Validasi (%)'])
        .highlight_max(subset=['Akurasi Validasi (%)'], color='#c6efce')
        .set_caption('Perbandingan CNN Kustom vs Transfer Learning')
    )
except:
    display(df_perbandingan)

## 15. Kesimpulan & Rekomendasi

### Hasil Eksperimen

| Model | Akurasi Validasi | Keterangan |
|:---|:---:|:---|
| **CNN Kustom (3 Blok Conv)** | ~XX% | Dibangun dari awal |
| **MobileNetV2 (Transfer Learning)** | ~XX% | Pretrained ImageNet |

### Analisis

1. **CNN Kustom** — Model berhasil mempelajari fitur visual sampah dari awal menggunakan arsitektur 3 blok konvolusi. Data augmentasi membantu mencegah overfitting.

2. **Transfer Learning (MobileNetV2)** — Memanfaatkan fitur yang telah dipelajari dari ImageNet mempercepat konvergensi dan umumnya menghasilkan akurasi lebih tinggi, terutama pada dataset yang lebih kecil.

3. **Kelas yang Sulit** — Kelas *trash* (sampah umum) cenderung memiliki akurasi lebih rendah karena variasi visual yang tinggi dan bisa menyerupai kelas lain.

### Rekomendasi

| Kondisi | Pilihan |
|:---|:---:|
| Dataset kecil, GPU terbatas | **Transfer Learning** |
| Dataset besar, kontrol penuh | **CNN Kustom** |
| Produksi / Deployment mobile | **MobileNetV2 + Fine-tuning** |

### Potensi Peningkatan
- Fine-tuning seluruh layer MobileNetV2 setelah fase pertama
- Penggunaan **EfficientNet** atau **ResNet50** sebagai backbone
- Menambah data dengan scraping atau augmentasi yang lebih agresif
- Implementasi **Grad-CAM** untuk interpretabilitas prediksi model